In [1]:
import sys; sys.path.append("..")
import numpy as np
import pandas as pd
from src.features import add_features, EQUIPMENT_VOCAB
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score

train = add_features(pd.read_csv("../data/processed/train.csv"))  # le test n'est PAS chargé
X_train = train.drop(columns="Price")
y_train = train["Price"]

num_cols = ["mileage_num", "car_age", "fiscal_power_num", "Number of Doors", "mileage_suspect"]
eq_cols  = ["eq_" + n for n in EQUIPMENT_VOCAB]
cat_low  = ["Condition", "Gearbox", "Fuel", "Origin", "First Owner"]
cat_high = ["Brand", "Model", "Location", "Sector"]

print(X_train.shape, len(eq_cols))

(59004, 36) 16


In [2]:
def make_pipeline(num, min_freq=30):
    prep = ColumnTransformer([
        ("num", SimpleImputer(strategy="median"), num),
        ("low", Pipeline([
            ("imp", SimpleImputer(strategy="constant", fill_value="Inconnu")),
            ("oh", OneHotEncoder(handle_unknown="ignore")),
        ]), cat_low),
        ("high", Pipeline([
            ("imp", SimpleImputer(strategy="constant", fill_value="Inconnu")),
            ("oh", OneHotEncoder(handle_unknown="ignore", min_frequency=min_freq)),
        ]), cat_high),
    ])
    return Pipeline([("prep", prep), ("reg", LinearRegression())])

scores = cross_val_score(
    make_pipeline(num_cols + eq_cols), X_train, y_train, cv=5,
    scoring="neg_mean_absolute_error", n_jobs=-1,
)
print("MAE par pli :", (-scores).round())
print("MAE (CV)    :", round(-scores.mean()))  # attendu : 44 634

MAE par pli : [44747. 45043. 44708. 44242. 44433.]
MAE (CV)    : 44634


In [3]:
import numpy as np
from sklearn.compose import TransformedTargetRegressor

log_model = TransformedTargetRegressor(
    regressor=make_pipeline(num_cols + eq_cols),
    func=np.log1p,
    inverse_func=np.expm1,
)

scores = cross_val_score(
    log_model, X_train, y_train, cv=5,
    scoring="neg_mean_absolute_error", n_jobs=-1,
)
mae_log = -scores

# Plis du test D (notebook 03)
mae_d = np.array([44747, 45043, 44708, 44242, 44433])

print("MAE par pli      :", mae_log.round())
print("MAE moyenne      :", round(mae_log.mean()))
print("Écart avec D     :", (mae_log - mae_d).round())

MAE par pli      : [34772. 34916. 34438. 33943. 33713.]
MAE moyenne      : 34357
Écart avec D     : [ -9975. -10127. -10270. -10299. -10720.]


In [4]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_validate

rf_pipeline = make_pipeline(num_cols + eq_cols)
rf_pipeline.set_params(reg=RandomForestRegressor(
    n_estimators=100, random_state=42, n_jobs=-1
))

rf_model = TransformedTargetRegressor(
    regressor=rf_pipeline,
    func=np.log1p,
    inverse_func=np.expm1,
)

res = cross_validate(
    rf_model, X_train, y_train, cv=5,
    scoring="neg_mean_absolute_error",
    return_train_score=True,
    verbose=2,
)

mae_rf = -res["test_score"]
mae_rf_train = -res["train_score"]

print("MAE validation par pli :", mae_rf.round())
print("MAE validation moyenne :", round(mae_rf.mean()))
print("MAE train moyenne      :", round(mae_rf_train.mean()))
print("Écart avec le linéaire :", (mae_rf - mae_log).round())

[CV] END .................................................... total time=14.8min
[CV] END .................................................... total time=15.2min
[CV] END .................................................... total time=14.1min
[CV] END .................................................... total time=23.8min
[CV] END .................................................... total time=16.0min
MAE validation par pli : [17829. 18165. 18007. 17427. 17821.]
MAE validation moyenne : 17850
MAE train moyenne      : 7251
Écart avec le linéaire : [-16943. -16751. -16431. -16517. -15892.]


[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed: 84.1min finished


In [5]:
same = X_train.duplicated(keep=False)
print("Lignes avec un jumeau (hors prix) :", same.sum())
print("Part du train :", round(same.mean() * 100, 2), "%")

Lignes avec un jumeau (hors prix) : 56
Part du train : 0.09 %


In [6]:
%pip install xgboost

   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.1/101.7 MB 2.0 MB/s eta 0:00:52
   ---------------------------------------- 0.2/101.7 MB 2.1 MB/s eta 0:00:48
   ---------------------------------------- 0.6/101.7 MB 3.6 MB/s eta 0:00:29
    --------------------------------------- 1.8/101.7 MB 6.3 MB/s eta 0:00:16
    --------------------------------------- 1.8/101.7 MB 6.2 MB/s eta 0:00:17
    --------------------------------------- 1.9/101.7 MB 4.7 MB/s eta 0:00:22
    --------------------------------------- 2.3/101.7 MB 4.8 MB/s eta 0:00:21
   - -------------------------------------- 3.9/101.7 MB 7.4 MB/s eta 0:00:14
   -- ------------------------------------- 5.2/101.7 MB 8.6 MB/s eta 0:00:12
   -- ------------------------------------- 5.2/101.7 MB 8.6 MB/s eta 0:00:12
   -- ------------------------------------- 6.0/101.7 MB 8.4 MB/s eta 0:00:12
   -- ------------------------------------- 6.5/101.7 MB 8.5 MB/s eta 0

ERROR: Exception:
Traceback (most recent call last):
  File "c:\Users\pp\mon_project\moroccan-used-car-price-prediction\.venv\Lib\site-packages\pip\_vendor\urllib3\response.py", line 438, in _error_catcher
    yield
  File "c:\Users\pp\mon_project\moroccan-used-car-price-prediction\.venv\Lib\site-packages\pip\_vendor\urllib3\response.py", line 561, in read
    data = self._fp_read(amt) if not fp_closed else b""
           ^^^^^^^^^^^^^^^^^^
  File "c:\Users\pp\mon_project\moroccan-used-car-price-prediction\.venv\Lib\site-packages\pip\_vendor\urllib3\response.py", line 527, in _fp_read
    return self._fp.read(amt) if amt is not None else self._fp.read()
           ^^^^^^^^^^^^^^^^^^
  File "c:\Users\pp\mon_project\moroccan-used-car-price-prediction\.venv\Lib\site-packages\pip\_vendor\cachecontrol\filewrapper.py", line 102, in read
    self.__buf.write(data)
  File "C:\Users\pp\AppData\Local\Programs\Python\Python311\Lib\tempfile.py", line 500, in func_wrapper
    return func(*args, **k

In [7]:
%pip install xgboost

  Using cached xgboost-3.2.0-py3-none-win_amd64.whl.metadata (2.1 kB)
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/101.7 MB 326.8 kB/s eta 0:05:12
   ---------------------------------------- 0.0/101.7 MB 326.8 kB/s eta 0:05:12
   ---------------------------------------- 0.1/101.7 MB 438.1 kB/s eta 0:03:52
   ---------------------------------------- 0.1/101.7 MB 438.1 kB/s eta 0:03:52
   ---------------------------------------- 0.1/101.7 MB 438.1 kB/s eta 0:03:52
   ---------------------------------------- 0.1/101.7 MB 438.1 kB/s eta 0:03:52
   ---------------------------------------- 0.1/101.7 MB 438.1 kB/s eta 0:03:52
   ---------------------------------------- 0.1/101.7 MB 438.1 kB/s eta 0:03:52
   ---------------------------------------- 0.2/101.7 MB 296.2 kB/s eta 0:05:43
   ---------------------------------------- 0.2/101.7 MB 296.2 kB/s 


[notice] A new release of pip is available: 24.0 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
from xgboost import XGBRegressor

xgb_pipeline = make_pipeline(num_cols + eq_cols)
xgb_pipeline.set_params(reg=XGBRegressor(
    n_estimators=500, learning_rate=0.1, max_depth=6,
    random_state=42, n_jobs=-1,
))

xgb_model = TransformedTargetRegressor(
    regressor=xgb_pipeline,
    func=np.log1p,
    inverse_func=np.expm1,
)

res_xgb = cross_validate(
    xgb_model, X_train, y_train, cv=5,
    scoring="neg_mean_absolute_error",
    return_train_score=True,
    verbose=2,
)

mae_xgb = -res_xgb["test_score"]

print("MAE validation par pli :", mae_xgb.round())
print("MAE validation moyenne :", round(mae_xgb.mean()))
print("MAE train moyenne      :", round(-res_xgb["train_score"].mean()))
print("Écart avec le RF       :", (mae_xgb - mae_rf).round())

[CV] END .................................................... total time=   5.5s
[CV] END .................................................... total time=   5.2s
[CV] END .................................................... total time=   4.9s
[CV] END .................................................... total time=   5.8s
[CV] END .................................................... total time=   5.1s
MAE validation par pli : [19814. 19892. 19687. 19505. 19709.]
MAE validation moyenne : 19721
MAE train moyenne      : 17426
Écart avec le RF       : [1986. 1727. 1680. 2078. 1887.]


[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:   31.9s finished


In [9]:
from sklearn.base import clone

xgb_d8 = clone(xgb_model).set_params(regressor__reg__max_depth=8)

res_d8 = cross_validate(
    xgb_d8, X_train, y_train, cv=5,
    scoring="neg_mean_absolute_error",
    return_train_score=True,
)

mae_d8 = -res_d8["test_score"]

print("MAE validation par pli   :", mae_d8.round())
print("MAE validation moyenne   :", round(mae_d8.mean()))
print("MAE train moyenne        :", round(-res_d8["train_score"].mean()))
print("Écart avec XGB profond. 6 :", (mae_d8 - mae_xgb).round())
print("Écart avec le RF          :", (mae_d8 - mae_rf).round())

MAE validation par pli   : [18935. 19007. 19003. 18586. 18989.]
MAE validation moyenne   : 18904
MAE train moyenne        : 14848
Écart avec XGB profond. 6 : [-879. -885. -684. -919. -720.]
Écart avec le RF          : [1106.  842.  996. 1159. 1167.]


In [10]:
xgb_d10 = clone(xgb_model).set_params(regressor__reg__max_depth=10)

res_d10 = cross_validate(
    xgb_d10, X_train, y_train, cv=5,
    scoring="neg_mean_absolute_error",
    return_train_score=True,
)

mae_d10 = -res_d10["test_score"]

print("MAE validation par pli   :", mae_d10.round())
print("MAE validation moyenne   :", round(mae_d10.mean()))
print("MAE train moyenne        :", round(-res_d10["train_score"].mean()))
print("Écart avec XGB profond. 8 :", (mae_d10 - mae_d8).round())
print("Écart avec le RF          :", (mae_d10 - mae_rf).round())

MAE validation par pli   : [18292. 18408. 18470. 17804. 18301.]
MAE validation moyenne   : 18255
MAE train moyenne        : 12163
Écart avec XGB profond. 8 : [-643. -599. -533. -782. -688.]
Écart avec le RF          : [463. 243. 463. 377. 479.]


In [11]:
xgb_d12 = clone(xgb_model).set_params(regressor__reg__max_depth=12)

res_d12 = cross_validate(
    xgb_d12, X_train, y_train, cv=5,
    scoring="neg_mean_absolute_error",
    return_train_score=True,
)

mae_d12 = -res_d12["test_score"]

print("MAE validation par pli    :", mae_d12.round())
print("MAE validation moyenne    :", round(mae_d12.mean()))
print("MAE train moyenne         :", round(-res_d12["train_score"].mean()))
print("Écart avec XGB profond. 10 :", (mae_d12 - mae_d10).round())
print("Écart avec le RF           :", (mae_d12 - mae_rf).round())

MAE validation par pli    : [17962. 17825. 18157. 17520. 17766.]
MAE validation moyenne    : 17846
MAE train moyenne         : 9670
Écart avec XGB profond. 10 : [-330. -583. -312. -284. -534.]
Écart avec le RF           : [ 133. -340.  150.   93.  -55.]


In [12]:
xgb_slow = clone(xgb_d12).set_params(
    regressor__reg__n_estimators=1000,
    regressor__reg__learning_rate=0.05,
)

res_slow = cross_validate(
    xgb_slow, X_train, y_train, cv=5,
    scoring="neg_mean_absolute_error",
    return_train_score=True,
)

mae_slow = -res_slow["test_score"]

print("MAE validation par pli :", mae_slow.round())
print("MAE validation moyenne :", round(mae_slow.mean()))
print("MAE train moyenne      :", round(-res_slow["train_score"].mean()))
print("Écart avec XGB d12     :", (mae_slow - mae_d12).round())

MAE validation par pli : [17804. 17791. 18069. 17471. 17712.]
MAE validation moyenne : 17769
MAE train moyenne      : 9754
Écart avec XGB d12     : [-158.  -34.  -89.  -49.  -55.]
